# 🧭 Elective Compass — Exploratory Data Analysis

**Author:** Paballo Precision Malepa  
**Project:** Elective Compass — A data-driven tool to help students navigate software specialisations  

---

## Overview

This notebook documents the full EDA process for the Elective Compass dataset. We work with:
- **7 course syllabi** — one per WeThinkCode elective
- **70 job descriptions** — 10 per elective, sourced from real job postings

The analysis answers five key questions:

1. **What does the raw data look like?** — corpus statistics and text length distributions
2. **What are the most important words per elective?** — TF-IDF keyword analysis
3. **How similar are electives to real-world jobs?** — Full 7×70 similarity heatmap
4. **How different are the electives from each other?** — Vocabulary overlap and cross-elective similarity
5. **What do jobs demand that syllabi don't teach?** — Skills gap analysis

---

## 1. Setup & Imports

In [ ]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

# ── Plot aesthetics ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

PALETTE = [
    '#d98aaa', '#8ab4d9', '#8ad9b0', '#d9c28a',
    '#b08ad9', '#d98a8a', '#8ad9d9'
]

print('Libraries loaded successfully.')

## 2. Load the Dataset

In [ ]:
# ── Resolve paths relative to this notebook ─────────────────────────────────
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
BASE_DIR = os.path.join(NOTEBOOK_DIR, '..')
DATA_DIR = os.path.join(BASE_DIR, 'data', 'raw')

ELECTIVE_LABELS = {
    'blockchain':         'Blockchain',
    'cloud':              'Cloud Computing',
    'cybersecurity':      'Cybersecurity',
    'data_eng':           'Data Engineering',
    'mobile':             'Mobile Dev',
    'qa':                 'Quality Assurance',
    'system_integration': 'System Integration',
}

def load_corpus():
    records = []

    # Syllabi
    for path in glob.glob(os.path.join(DATA_DIR, 'syllabus_*.txt')):
        key = os.path.basename(path).replace('syllabus_', '').replace('.txt', '')
        label = ELECTIVE_LABELS.get(key, key.replace('_', ' ').title())
        text = open(path, encoding='utf-8').read()
        records.append({'doc_type': 'syllabus', 'elective_key': key,
                         'elective': label, 'filename': os.path.basename(path), 'text': text})

    # Job descriptions
    for path in glob.glob(os.path.join(DATA_DIR, 'jobs', 'job_*.txt')):
        basename = os.path.basename(path).replace('.txt', '')  # e.g. job_cloud_3
        parts = basename.split('_')
        # key = everything between 'job' and the trailing number
        key = '_'.join(parts[1:-1])
        label = ELECTIVE_LABELS.get(key, key.replace('_', ' ').title())
        text = open(path, encoding='utf-8').read()
        records.append({'doc_type': 'job', 'elective_key': key,
                         'elective': label, 'filename': os.path.basename(path), 'text': text})

    return pd.DataFrame(records)

df = load_corpus()

print(f'Total documents loaded : {len(df)}')
print(f'  Syllabi              : {(df.doc_type == "syllabus").sum()}')
print(f'  Job descriptions     : {(df.doc_type == "job").sum()}')
print()
print('Documents per elective:')
print(df.groupby(['elective', 'doc_type']).size().unstack(fill_value=0).to_string())

## 3. Corpus Statistics

Before any modelling, we get a feel for the data: document lengths, vocabulary size, and how much text we actually have.

In [ ]:
# ── Word count per document ──────────────────────────────────────────────────
df['word_count'] = df['text'].apply(lambda t: len(t.split()))
df['char_count'] = df['text'].apply(len)

summary = df.groupby(['doc_type'])['word_count'].agg(['mean', 'min', 'max', 'sum'])
summary.columns = ['Mean words', 'Min words', 'Max words', 'Total words']
summary.index = ['Job descriptions', 'Syllabi']
print('=== Word Count Summary ===')
print(summary.round(0).astype(int).to_string())

In [ ]:
# ── Word count distribution per elective (jobs only) ────────────────────────
jobs_df = df[df['doc_type'] == 'job'].copy()

fig, ax = plt.subplots(figsize=(12, 5))

elective_order = sorted(jobs_df['elective'].unique())
data_to_plot = [jobs_df[jobs_df['elective'] == e]['word_count'].values for e in elective_order]

bp = ax.boxplot(data_to_plot, patch_artist=True, notch=False,
                medianprops=dict(color='#4a3040', linewidth=2))

for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticklabels(elective_order, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Word count', fontsize=11)
ax.set_title('Distribution of Job Description Word Counts per Elective', fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

**Observations:**  
- Cybersecurity job postings tend to be the longest — they list detailed compliance requirements and toolchains.  
- Blockchain and Mobile postings are typically shorter and more focused.  
- Syllabi are significantly shorter than job descriptions, which is expected — they summarise a course rather than enumerate every skill.

## 4. Top Word Frequencies per Elective

We use raw term frequency (CountVectorizer) to find the most common words in the job descriptions for each elective, after removing English stopwords.

In [ ]:
def top_words_for_elective(elective_label, n=15, doc_type='job'):
    """Return a DataFrame of (word, count) for the top n words in an elective's documents."""
    subset = df[(df['elective'] == elective_label) & (df['doc_type'] == doc_type)]
    corpus = subset['text'].tolist()
    cv = CountVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 1))
    X = cv.fit_transform(corpus)
    word_counts = np.array(X.sum(axis=0)).flatten()
    words = cv.get_feature_names_out()
    top_idx = word_counts.argsort()[::-1][:n]
    return pd.DataFrame({'word': words[top_idx], 'count': word_counts[top_idx]})


# ── Plot top-15 words for each elective ─────────────────────────────────────
electives = sorted(jobs_df['elective'].unique())
fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.flatten()

for i, (elective, color) in enumerate(zip(electives, PALETTE)):
    top = top_words_for_elective(elective, n=15)
    ax = axes[i]
    bars = ax.barh(top['word'][::-1], top['count'][::-1], color=color, alpha=0.85)
    ax.set_title(elective, fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency', fontsize=9)
    ax.tick_params(labelsize=9)
    for bar in bars:
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                f'{int(bar.get_width())}', va='center', fontsize=7.5)

# Hide last empty subplot if odd count
if len(electives) < len(axes):
    axes[-1].set_visible(False)

fig.suptitle('Top 15 Most Frequent Words in Job Descriptions (per Elective)',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Observations:**  
- "experience", "knowledge", and "understanding" appear across all electives — these are generic job posting terms, not technical signals.  
- Domain-specific terms like *solidity*, *smart*, *contracts* (Blockchain), *kafka*, *pipeline*, *airflow* (Data Engineering), and *flutter*, *firebase* (Mobile) clearly distinguish electives.  
- This motivates the use of TF-IDF over raw counts — it down-weights the common terms and surfaces the truly distinctive vocabulary.

## 5. TF-IDF Keyword Extraction per Elective

TF-IDF (Term Frequency–Inverse Document Frequency) weights words that are important within one elective's documents but rare across all electives. These are the most *distinctive* terms per specialisation.

In [ ]:
# ── Aggregate all job descriptions per elective into one document ────────────
elective_corpus = (
    jobs_df.groupby('elective')['text']
    .apply(lambda texts: ' '.join(texts))
    .reset_index()
)
elective_corpus.columns = ['elective', 'combined_text']

# Fit TF-IDF treating each elective as one document
tfidf = TfidfVectorizer(stop_words='english', max_features=3000, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(elective_corpus['combined_text'])
feature_names = tfidf.get_feature_names_out()

def top_tfidf_terms(elective_label, n=12):
    """Return top n TF-IDF terms for a given elective."""
    idx = elective_corpus[elective_corpus['elective'] == elective_label].index[0]
    scores = tfidf_matrix[idx].toarray().flatten()
    top_idx = scores.argsort()[::-1][:n]
    return pd.DataFrame({'term': feature_names[top_idx], 'tfidf_score': scores[top_idx]})


# ── Plot TF-IDF keywords ─────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.flatten()

for i, (elective, color) in enumerate(zip(elective_corpus['elective'].tolist(), PALETTE)):
    top = top_tfidf_terms(elective, n=12)
    ax = axes[i]
    bars = ax.barh(top['term'][::-1], top['tfidf_score'][::-1], color=color, alpha=0.85)
    ax.set_title(f'{elective} — TF-IDF Keywords', fontsize=11, fontweight='bold')
    ax.set_xlabel('TF-IDF Score', fontsize=9)
    ax.tick_params(labelsize=9)
    for bar in bars:
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                f'{bar.get_width():.3f}', va='center', fontsize=7.5)

if len(elective_corpus) < len(axes):
    axes[-1].set_visible(False)

fig.suptitle('Top 12 TF-IDF Keywords per Elective\n(Distinctive terms — common words down-weighted)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Observations:**  
- The TF-IDF keywords are far more informative than raw counts. *Vyper*, *ERC20*, *dApps* clearly signal Blockchain. *Kafka*, *Airflow*, *Spark* distinguish Data Engineering.  
- Bigrams (two-word phrases) like *smart contracts*, *test automation*, *message queue* are captured by the `ngram_range=(1,2)` setting and are often more semantically precise than single words.  
- This TF-IDF representation forms the core of the similarity engine in the Elective Compass app.

## 6. Full Similarity Heatmap: All Syllabi × All Jobs

We compute cosine similarity between every syllabus and every job description to produce a 7×70 similarity matrix.

In [ ]:
# ── Build TF-IDF matrix over the full corpus ─────────────────────────────────
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
all_texts = df['text'].tolist()
full_tfidf = vectorizer.fit_transform(all_texts)

syllabus_mask = df['doc_type'] == 'syllabus'
job_mask      = df['doc_type'] == 'job'

syllabus_vectors = full_tfidf[df[syllabus_mask].index]
job_vectors      = full_tfidf[df[job_mask].index]

sim_matrix = cosine_similarity(syllabus_vectors, job_vectors)  # shape: (7, 70)

syllabus_labels = df[syllabus_mask]['elective'].tolist()
job_labels      = df[job_mask]['filename'].apply(
    lambda f: f.replace('job_', '').replace('.txt', '').replace('_', ' ')
).tolist()

sim_df = pd.DataFrame(sim_matrix, index=syllabus_labels, columns=job_labels)

print(f'Similarity matrix shape: {sim_df.shape}  (syllabi × jobs)')
print(f'Score range: {sim_df.values.min():.3f} – {sim_df.values.max():.3f}')

In [ ]:
# ── Full 7×70 heatmap ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(22, 6))

# Sort job columns by elective for a clean block structure
job_order = df[job_mask].sort_values('elective')['filename'].apply(
    lambda f: f.replace('job_', '').replace('.txt', '').replace('_', ' ')
).tolist()
sim_df_sorted = sim_df[job_order]

cmap = sns.color_palette('RdPu', as_cmap=True)
sns.heatmap(
    sim_df_sorted,
    ax=ax,
    cmap=cmap,
    annot=False,
    linewidths=0.3,
    linecolor='#f5e6ee',
    cbar_kws={'label': 'Cosine Similarity', 'shrink': 0.7}
)

ax.set_xlabel('Job Descriptions (sorted by elective)', fontsize=11)
ax.set_ylabel('Course Syllabus', fontsize=11)
ax.set_title('Cosine Similarity: All Syllabi vs All Job Descriptions',
             fontsize=14, fontweight='bold', pad=16)
ax.tick_params(axis='x', labelrotation=90, labelsize=6.5)
ax.tick_params(axis='y', labelrotation=0, labelsize=10)
plt.tight_layout()
plt.show()

**Observations:**  
- The diagonal blocks (where each syllabus row meets its own elective's job columns) are visibly the darkest, confirming the model correctly identifies within-elective alignment.  
- Cloud Computing and Data Engineering share visible overlap — both involve AWS, Python, and cloud infrastructure.  
- Blockchain shows high specificity: it matches well to Blockchain jobs and poorly to everything else.  
- QA has the most spread-out scores — quality assurance skills (testing, Python scripting) are broadly applicable.

## 7. Cross-Elective Similarity: How Different Are the Electives From Each Other?

A 7×7 matrix comparing each elective's syllabus against every other elective's syllabus. High similarity means the electives teach overlapping skills — important for students choosing between them.

In [ ]:
# ── 7×7 cross-syllabus similarity ───────────────────────────────────────────
cross_sim = cosine_similarity(syllabus_vectors, syllabus_vectors)
cross_df = pd.DataFrame(cross_sim, index=syllabus_labels, columns=syllabus_labels)

# Mask the diagonal (self-similarity = 1.0) for cleaner reading
np.fill_diagonal(cross_sim, np.nan)
cross_df_masked = pd.DataFrame(cross_sim, index=syllabus_labels, columns=syllabus_labels)

fig, ax = plt.subplots(figsize=(9, 7))

sns.heatmap(
    cross_df_masked,
    ax=ax,
    cmap='RdPu',
    annot=True,
    fmt='.2f',
    linewidths=1,
    linecolor='#f5e6ee',
    vmin=0, vmax=0.5,
    cbar_kws={'label': 'Cosine Similarity', 'shrink': 0.8},
    square=True
)

ax.set_title('Cross-Elective Syllabus Similarity\n(How much do electives overlap?)',
             fontsize=13, fontweight='bold', pad=16)
ax.tick_params(axis='x', labelrotation=30, labelsize=9)
ax.tick_params(axis='y', labelrotation=0, labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Most and least similar elective pairs ───────────────────────────────────
pairs = []
for i, a in enumerate(syllabus_labels):
    for j, b in enumerate(syllabus_labels):
        if j > i:
            pairs.append({'Elective A': a, 'Elective B': b,
                          'Similarity': round(float(cross_df.iloc[i, j]), 4)})

pairs_df = pd.DataFrame(pairs).sort_values('Similarity', ascending=False)

print('=== Most Similar Elective Pairs ===')
print(pairs_df.head(4).to_string(index=False))
print()
print('=== Most Distinct Elective Pairs ===')
print(pairs_df.tail(4).to_string(index=False))

**Observations:**  
- Cloud Computing and Data Engineering tend to be the most similar electives — both syllabi reference Python, AWS, and scalable systems.  
- Blockchain is consistently the most distinct — its vocabulary (*Solidity*, *NFTs*, *DeFi*, *Vyper*) appears nowhere else.  
- This matrix is directly useful for confused students: if you're unsure between Cloud and Data Engineering, the high similarity score tells you they share foundational skills before diverging.

## 8. Vocabulary Overlap Analysis

How much of each elective's vocabulary appears in other electives? We measure this as a Jaccard similarity over the top-200 TF-IDF terms per elective.

In [ ]:
def get_top_vocab(elective_label, n=200, doc_type='job'):
    """Return a set of the top n TF-IDF unigrams for an elective."""
    subset = df[(df['elective'] == elective_label) & (df['doc_type'] == doc_type)]
    corpus = subset['text'].tolist()
    cv = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 1))
    X = cv.fit_transform(corpus)
    scores = np.array(X.sum(axis=0)).flatten()
    words = cv.get_feature_names_out()
    top_idx = scores.argsort()[::-1][:n]
    return set(words[top_idx])


elective_names = sorted(jobs_df['elective'].unique())
vocab_sets = {e: get_top_vocab(e, n=200) for e in elective_names}

# Jaccard similarity matrix
n = len(elective_names)
jaccard_matrix = np.zeros((n, n))

for i, a in enumerate(elective_names):
    for j, b in enumerate(elective_names):
        if i == j:
            jaccard_matrix[i, j] = 1.0
        else:
            inter = len(vocab_sets[a] & vocab_sets[b])
            union = len(vocab_sets[a] | vocab_sets[b])
            jaccard_matrix[i, j] = inter / union if union > 0 else 0

jaccard_df = pd.DataFrame(jaccard_matrix, index=elective_names, columns=elective_names)

# Mask diagonal
np.fill_diagonal(jaccard_matrix, np.nan)
jaccard_df_masked = pd.DataFrame(jaccard_matrix, index=elective_names, columns=elective_names)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    jaccard_df_masked,
    ax=ax,
    cmap='RdPu',
    annot=True,
    fmt='.2f',
    linewidths=1,
    linecolor='#f5e6ee',
    vmin=0, vmax=0.4,
    cbar_kws={'label': 'Jaccard Similarity', 'shrink': 0.8},
    square=True
)

ax.set_title('Vocabulary Overlap Between Electives\n(Jaccard Similarity of Top-200 TF-IDF Terms)',
             fontsize=13, fontweight='bold', pad=16)
ax.tick_params(axis='x', labelrotation=30, labelsize=9)
ax.tick_params(axis='y', labelrotation=0, labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ── Unique vocabulary size per elective ─────────────────────────────────────
all_other_vocab = {e: set().union(*[vocab_sets[o] for o in elective_names if o != e])
                   for e in elective_names}

uniqueness_data = []
for e in elective_names:
    vocab = vocab_sets[e]
    unique_terms = vocab - all_other_vocab[e]
    uniqueness_data.append({
        'Elective': e,
        'Top-200 vocab': len(vocab),
        'Unique to elective': len(unique_terms),
        'Uniqueness %': round(len(unique_terms) / len(vocab) * 100, 1),
        'Sample unique terms': ', '.join(sorted(unique_terms)[:5])
    })

uniqueness_df = pd.DataFrame(uniqueness_data).sort_values('Uniqueness %', ascending=False)
print('=== Vocabulary Uniqueness per Elective ===')
print(uniqueness_df.to_string(index=False))

**Observations:**  
- Blockchain has the highest vocabulary uniqueness — its technical terms simply don't appear in any other elective.  
- QA and System Integration share the most vocabulary with other electives, suggesting these roles have transferable skills across the other domains.  
- This uniqueness score could directly power an "elective distinctiveness" UI feature for students.

## 9. Skills Gap Analysis

**The core question for students:** *What does industry demand that the syllabus doesn't mention?*

We extract the top TF-IDF terms from job descriptions for each elective, then check which terms are absent from the corresponding syllabus. The gap terms are skills students may need to self-develop.

In [ ]:
def skills_gap(elective_label, top_n_job_terms=30, min_syllabus_overlap=2):
    """
    For a given elective:
    - Takes the top N TF-IDF terms from all job descriptions
    - Checks which are absent (or near-absent) from the syllabus
    - Returns a sorted DataFrame of gap terms with their job-frequency scores
    """
    # Job corpus for the elective
    job_texts = df[(df['elective'] == elective_label) & (df['doc_type'] == 'job')]['text'].tolist()
    # Syllabus text
    syllabus_text = df[(df['elective'] == elective_label) & (df['doc_type'] == 'syllabus')]['text'].values
    if len(syllabus_text) == 0:
        return pd.DataFrame()
    syllabus_words = set(syllabus_text[0].lower().split())

    # TF-IDF on job texts
    tv = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 1))
    X = tv.fit_transform(job_texts)
    scores = np.array(X.sum(axis=0)).flatten()
    words = tv.get_feature_names_out()

    top_idx = scores.argsort()[::-1][:top_n_job_terms]
    top_job_terms = [(words[i], scores[i]) for i in top_idx]

    gaps = []
    for word, score in top_job_terms:
        # Count rough occurrences in syllabus (case-insensitive)
        syllabus_occurrences = syllabus_text[0].lower().count(word)
        if syllabus_occurrences < min_syllabus_overlap:
            gaps.append({'skill': word, 'job_demand_score': round(float(score), 4),
                         'syllabus_mentions': syllabus_occurrences})

    return pd.DataFrame(gaps)


# ── Run gap analysis for all electives ──────────────────────────────────────
print('=== Skills Gap Analysis — Top Demanded Skills Missing from Syllabi ===\n')
for elective in elective_names:
    gap_df = skills_gap(elective, top_n_job_terms=25, min_syllabus_overlap=1)
    top_gaps = gap_df.head(8)
    print(f'--- {elective} ---')
    if top_gaps.empty:
        print('  No significant gaps detected.')
    else:
        for _, row in top_gaps.iterrows():
            bar = '█' * int(row['job_demand_score'] * 150)
            print(f"  {row['skill']:<25} {bar}  (score: {row['job_demand_score']:.3f}, syllabus mentions: {row['syllabus_mentions']})")
    print()

In [ ]:
# ── Visual skills gap chart ──────────────────────────────────────────────────
fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.flatten()

for i, (elective, color) in enumerate(zip(elective_names, PALETTE)):
    gap_df = skills_gap(elective, top_n_job_terms=30, min_syllabus_overlap=1).head(10)
    ax = axes[i]
    if gap_df.empty:
        ax.text(0.5, 0.5, 'No gaps', ha='center', va='center')
    else:
        bars = ax.barh(gap_df['skill'][::-1], gap_df['job_demand_score'][::-1],
                       color=color, alpha=0.85)
        ax.set_title(f'{elective} — Skills Gap', fontsize=11, fontweight='bold')
        ax.set_xlabel('Job Demand Score (TF-IDF)', fontsize=9)
        ax.tick_params(labelsize=9)

if len(elective_names) < len(axes):
    axes[-1].set_visible(False)

fig.suptitle('Skills Gap: Top In-Demand Skills NOT Mentioned in Syllabi\n(What students may need to self-develop)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Observations:**  
- Cloud Computing has a notable gap around specific tools like *Terraform*, *Kubernetes*, and *monitoring* — the syllabus covers AWS concepts but industry expects hands-on infra-as-code skills.  
- Data Engineering's gap includes *dbt*, *delta lake*, and *orchestration* — modern data stack tools not yet in the curriculum.  
- Cybersecurity shows a gap for *SIEM*, *Active Directory*, and *compliance frameworks* — real-world blue team skills beyond the introductory TryHackMe path.  
- These gaps are valuable signal for both students (self-study plan) and curriculum designers (future updates).

## 10. Best Elective per Job Role

Which elective best prepares students for each specific job title? We find the highest-similarity syllabus for every job description.

In [ ]:
# Transpose the similarity matrix: rows = jobs, find best syllabus
best_match = sim_df.idxmax(axis=0).reset_index()
best_match.columns = ['job', 'best_elective']
best_match['score'] = sim_df.max(axis=0).values

# Add the actual elective category of the job
job_elective_map = df[job_mask][['filename', 'elective']].copy()
job_elective_map['filename'] = job_elective_map['filename'].apply(
    lambda f: f.replace('job_', '').replace('.txt', '').replace('_', ' ')
)
best_match = best_match.merge(job_elective_map.rename(columns={'filename': 'job', 'elective': 'job_category'}),
                               on='job', how='left')

best_match['correct_match'] = best_match['best_elective'] == best_match['job_category']

accuracy = best_match['correct_match'].mean() * 100
print(f'Model correctly matched {accuracy:.1f}% of job descriptions to their own elective\'s syllabus.')
print()

# Show mismatched jobs (interesting edge cases)
mismatches = best_match[~best_match['correct_match']]
if len(mismatches) > 0:
    print(f'=== {len(mismatches)} Mismatched Jobs (Cross-Domain Skills) ===')
    print(mismatches[['job', 'job_category', 'best_elective', 'score']].to_string(index=False))
else:
    print('All job descriptions were correctly matched to their own elective.')

In [ ]:
# ── Accuracy summary by elective ─────────────────────────────────────────────
accuracy_by_elective = (
    best_match.groupby('job_category')['correct_match']
    .agg(['sum', 'count'])
    .rename(columns={'sum': 'correct', 'count': 'total'})
)
accuracy_by_elective['accuracy_pct'] = (accuracy_by_elective['correct'] / accuracy_by_elective['total'] * 100).round(1)
accuracy_by_elective = accuracy_by_elective.sort_values('accuracy_pct', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [PALETTE[i % len(PALETTE)] for i in range(len(accuracy_by_elective))]
bars = ax.bar(accuracy_by_elective.index, accuracy_by_elective['accuracy_pct'],
              color=colors, alpha=0.85, edgecolor='white', linewidth=1.5)

ax.axhline(100, color='#4a3040', linestyle='--', alpha=0.4, linewidth=1)
ax.set_ylim(0, 115)
ax.set_ylabel('Match Accuracy (%)', fontsize=11)
ax.set_title('TF-IDF Model Accuracy: Correct Syllabus–Job Matching by Elective',
             fontsize=13, fontweight='bold', pad=14)
ax.tick_params(axis='x', labelrotation=20, labelsize=10)

for bar, val in zip(bars, accuracy_by_elective['accuracy_pct']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{val:.0f}%', ha='center', va='bottom', fontsize=10, fontweight='bold', color='#4a3040')

plt.tight_layout()
plt.show()

**Observations:**  
- The TF-IDF model achieves strong matching accuracy for electives with distinctive vocabulary (Blockchain, Mobile, Data Engineering).  
- Lower accuracy on cross-domain electives (Cloud ↔ Data Engineering, QA ↔ System Integration) reflects genuine skill overlap — these aren't model failures but real-world ambiguity.  
- This accuracy metric gives a quantitative evaluation of the model used in the Elective Compass app.

## 11. Summary & Key Findings

| Finding | Insight |
|---|---|
| **Corpus size** | 7 syllabi + 70 job descriptions; job descriptions average ~200–300 words |
| **Most distinctive elective** | Blockchain — highest vocabulary uniqueness, lowest cross-elective similarity |
| **Most overlapping pair** | Cloud Computing ↔ Data Engineering — shared AWS, Python, and data tooling |
| **Largest skills gaps** | Cloud: Terraform/Kubernetes; Data Eng: dbt/Delta Lake; Cybersecurity: SIEM/Active Directory |
| **Model accuracy** | TF-IDF cosine similarity correctly matches the majority of jobs to their own elective |
| **Most transferable elective** | QA — skills generalise broadly across most other technical roles |

---

### What this means for students

- If you want a **highly specialised, niche career path** → Blockchain (most unique skills)
- If you want **maximum career flexibility** → Cloud or Data Engineering (broad demand, significant overlap)
- If you are **security-minded and enjoy breaking things** → Cybersecurity (clear domain identity)
- If you want **product-visible work on mobile devices** → Mobile Development
- If you value **rigour, process, and software quality** → Quality Assurance
- If you are drawn to **connecting systems and distributed architecture** → System Integration

---

*This notebook is the analytical backbone of the Elective Compass Streamlit app.*

---

## 12. Real Student Survey Analysis

**A third data source beyond syllabi and job descriptions.**

We collected survey responses from current and former WeThinkCode_ students asking:
- Which elective they chose and how satisfied they are with that choice
- Whether they would choose the same elective again
- How confident they felt when making their original decision
- What they find most satisfying about writing software

This section answers three questions:

1. **Do students regret their choice?** — satisfaction and would-choose-again analysis
2. **How informed were students when choosing?** — decision confidence histogram
3. **How should the quiz weights be calibrated?** — satisfaction-based weight multipliers

> **Note:** The survey uses *Data Science* as an elective label (the correct WeThinkCode_ name).
> The NLP model uses *Data Engineering* for the syllabus/job corpus. Where mapping is needed,
> Data Science survey data is mapped to the Data Engineering model key.

In [ ]:
# ── Load survey data ──────────────────────────────────────────────────────────
survey_path = os.path.join(DATA_DIR, 'survey.csv')
survey = pd.read_csv(survey_path)
survey['satisfaction']    = pd.to_numeric(survey['satisfaction'],    errors='coerce')
survey['had_enough_info'] = pd.to_numeric(survey['had_enough_info'], errors='coerce')
survey['elective']        = survey['elective'].str.strip()
survey['would_choose_again'] = survey['would_choose_again'].str.strip()

print(f'Survey rows   : {len(survey)}')
print(f'Electives     : {sorted(survey["elective"].unique())}')
print(f'Avg satisfaction  : {survey["satisfaction"].mean():.2f} / 5')
print(f'Avg info confidence: {survey["had_enough_info"].mean():.2f} / 5')
print()
print('Would choose again:')
print(survey['would_choose_again'].value_counts().to_string())

### 12.1 Satisfaction by Elective

In [ ]:
# ── Satisfaction by elective ──────────────────────────────────────────────────
sat_by_elective = (
    survey.groupby('elective')['satisfaction']
    .agg(['mean', 'median', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean':'avg', 'median':'med', 'std':'sd', 'count':'n'})
    .sort_values('avg', ascending=False)
)

SURVEY_COLORS = {
    'Cloud Computing':    '#2563eb',
    'Mobile Development': '#0d9488',
    'Data Science':       '#d97706',
    'Cybersecurity':      '#dc2626',
    'System Integration': '#0891b2',
    'Quality Assurance':  '#16a34a',
}

overall_avg = survey['satisfaction'].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: mean bar
ax = axes[0]
colors = [SURVEY_COLORS.get(e, '#94a3b8') for e in sat_by_elective['elective']]
bars = ax.barh(sat_by_elective['elective'], sat_by_elective['avg'],
               color=colors, alpha=0.85)
ax.axvline(overall_avg, color='#64748b', linestyle='--', linewidth=1.5,
           label=f'Overall avg ({overall_avg:.2f})')
ax.set_xlim(0, 5.8)
ax.set_xlabel('Average Satisfaction (1–5)', fontsize=11)
ax.set_title('Mean Satisfaction by Elective', fontsize=12, fontweight='bold')
for bar, row in zip(bars, sat_by_elective.itertuples()):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            f'{row.avg:.2f}  (n={row.n})', va='center', fontsize=9)
ax.legend(fontsize=9)

# Right: box plots per elective
ax2 = axes[1]
elective_order = sat_by_elective['elective'].tolist()
box_data = [survey[survey['elective'] == e]['satisfaction'].dropna().values
            for e in elective_order]
bp = ax2.boxplot(box_data, patch_artist=True, vert=True,
                 medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax2.set_xticks(range(1, len(elective_order) + 1))
ax2.set_xticklabels(elective_order, rotation=20, ha='right', fontsize=9)
ax2.set_ylabel('Satisfaction Score', fontsize=11)
ax2.set_ylim(0, 6)
ax2.set_title('Satisfaction Distribution per Elective', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(sat_by_elective.to_string(index=False))

**Observations:**
- Mobile Development has the highest average satisfaction (5.0/5), but only 2 respondents — treat with caution.
- Cybersecurity and System Integration students are broadly satisfied (4.3 and 4.25 respectively).
- Data Science has the most spread — some students love it, some rate it 2 or 3, indicating higher variance in expectations vs. reality.
- No elective averages below 3.9, which is a strong signal that WeThinkCode_ elective design is broadly well-received.

### 12.2 Would-Choose-Again Analysis

In [ ]:
# ── Would choose again — overall and by elective ─────────────────────────────
choice_counts = survey['would_choose_again'].value_counts()

choice_by_elective = (
    survey.groupby(['elective', 'would_choose_again'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for col in ['Yes', 'No', 'Not sure']:
    if col not in choice_by_elective.columns:
        choice_by_elective[col] = 0

# Compute % yes per elective
choice_by_elective['total'] = choice_by_elective[['Yes','No','Not sure']].sum(axis=1)
choice_by_elective['pct_yes'] = choice_by_elective['Yes'] / choice_by_elective['total'] * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: donut
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    choice_counts.values,
    labels=choice_counts.index,
    autopct='%1.0f%%',
    colors=['#16a34a', '#dc2626', '#d97706'],
    startangle=90,
    wedgeprops=dict(width=0.55)
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
ax.set_title('Would You Choose the Same Elective Again?', fontsize=12, fontweight='bold')

# Right: stacked bar by elective
ax2 = axes[1]
x = range(len(choice_by_elective))
ax2.bar(x, choice_by_elective['Yes'],      label='Yes',      color='#16a34a', alpha=0.85)
ax2.bar(x, choice_by_elective['No'],       bottom=choice_by_elective['Yes'],
        label='No', color='#dc2626', alpha=0.85)
ax2.bar(x, choice_by_elective['Not sure'],
        bottom=choice_by_elective['Yes'] + choice_by_elective['No'],
        label='Not sure', color='#d97706', alpha=0.85)
ax2.set_xticks(list(x))
ax2.set_xticklabels(choice_by_elective['elective'], rotation=20, ha='right', fontsize=9)
ax2.set_ylabel('Number of Students')
ax2.set_title('Would Choose Again — Breakdown by Elective', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print('% who would choose again per elective:')
print(choice_by_elective[['elective','Yes','No','Not sure','pct_yes']].to_string(index=False))

**Observations:**
- 75% of students overall would pick the same elective again — solid retention signal.
- Data Science has the most "No" responses, consistent with its higher satisfaction variance.
- Cloud Computing students who said "No" mostly preferred Blockchain or Data Science — adjacent specialisations.
- This directly validates the need for Elective Compass: the 25% who would switch shows real decision regret that better upfront guidance could reduce.

### 12.3 Information Confidence Before Choosing

In [ ]:
# ── Information confidence distribution ──────────────────────────────────────
info_dist = survey['had_enough_info'].value_counts().sort_index()
avg_info  = survey['had_enough_info'].mean()
low_conf  = (survey['had_enough_info'] <= 2).sum()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: histogram by level
ax = axes[0]
conf_colors = ['#dc2626', '#f97316', '#d97706', '#2563eb', '#16a34a']
bars = ax.bar(info_dist.index.astype(str), info_dist.values,
              color=conf_colors[:len(info_dist)], alpha=0.88, edgecolor='white')
ax.axhline(info_dist.values.mean(), color='#64748b', linestyle='--', linewidth=1.5,
           label='Mean count')
ax.set_xlabel('Confidence Level (1=Low, 5=High)', fontsize=11)
ax.set_ylabel('Number of Students', fontsize=11)
ax.set_title('How Confident Were Students Before Choosing?', fontsize=12, fontweight='bold')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'{int(bar.get_height())}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, info_dist.max() + 2)
ax.legend(fontsize=9)

# Right: confidence by elective (mean bar)
ax2 = axes[1]
conf_by_e = (
    survey.groupby('elective')['had_enough_info']
    .mean().sort_values(ascending=False)
)
colors_conf = [SURVEY_COLORS.get(e, '#94a3b8') for e in conf_by_e.index]
bars2 = ax2.barh(conf_by_e.index, conf_by_e.values, color=colors_conf, alpha=0.85)
ax2.axvline(avg_info, color='#64748b', linestyle='--', linewidth=1.5,
            label=f'Overall avg ({avg_info:.2f})')
ax2.set_xlim(0, 6.5)
ax2.set_xlabel('Avg Confidence Score (1–5)', fontsize=11)
ax2.set_title('Avg Pre-Choice Confidence by Elective', fontsize=12, fontweight='bold')
for bar, val in zip(bars2, conf_by_e.values):
    ax2.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=9)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'Average pre-choice confidence: {avg_info:.2f} / 5')
print(f'Students with confidence ≤ 2 : {low_conf} ({low_conf/len(survey)*100:.0f}%)')
print(f'Students with confidence ≥ 4 : {(survey["had_enough_info"] >= 4).sum()} ({(survey["had_enough_info"] >= 4).sum()/len(survey)*100:.0f}%)')

**Observations:**
- Average pre-choice confidence is only **3.29 / 5** — students generally felt under-informed.
- Over a third of students (≈36%) rated their confidence at 2 or below when they chose. This is the core problem Elective Compass is built to solve.
- System Integration students had the highest confidence (5.0 average) — possibly because the elective name is more self-explanatory.
- Mobile Development had the lowest pre-choice confidence (2.0 average) despite 100% satisfaction post-choice, suggesting students discovered it was right for them after choosing, not before.

### 12.4 What Students Find Most Satisfying

In [ ]:
# ── Satisfying task types ─────────────────────────────────────────────────────
task_counts = survey['satisfying_task'].dropna().value_counts()

# Shorten labels
short = {
    "Making sure things work correctly and don't break in production":
        "Making sure things work / don't break",
    "Making things scale reliably (infrastructure performance uptime)":
        "Making things scale reliably",
    "Getting separate systems/services to work together smoothly":
        "Getting systems to work together",
    "Making something visual and interactive that people directly use":
        "Visual & interactive output",
    "Turning raw data into something useful or understandable":
        "Turning raw data into insights",
    "Finding and closing security gaps before someone else finds them":
        "Finding & closing security gaps",
}
task_labels = [short.get(t, t[:50] + '…' if len(t) > 50 else t) for t in task_counts.index]

fig, ax = plt.subplots(figsize=(12, 5))
task_colors = ['#2563eb','#0d9488','#d97706','#dc2626','#7c3aed','#0891b2','#16a34a']
bars = ax.barh(task_labels[::-1], task_counts.values[::-1],
               color=task_colors[:len(task_counts)], alpha=0.88)
ax.set_xlabel('Number of Students', fontsize=11)
ax.set_title('What Students Find Most Satisfying About Coding', fontsize=13, fontweight='bold', pad=14)
ax.set_xlim(0, task_counts.max() + 2)
for bar in bars:
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width())}', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

# Cross-tab: satisfaction by satisfying task type
cross_task_sat = (
    survey.groupby('satisfying_task')['satisfaction']
    .agg(['mean', 'count'])
    .reset_index()
    .rename(columns={'mean': 'avg_sat', 'count': 'n'})
    .sort_values('avg_sat', ascending=False)
)
cross_task_sat['task_short'] = cross_task_sat['satisfying_task'].apply(
    lambda t: short.get(t, t[:45]+'…' if len(str(t)) > 45 else str(t))
)
print('=== Avg Satisfaction by Preferred Task Type ===')
print(cross_task_sat[['task_short','avg_sat','n']].to_string(index=False))

**Observations:**
- "Making sure things work / don't break" is the most common satisfying task — this maps to both QA and Cybersecurity orientations.
- "Turning raw data into insights" and "Making things scale reliably" are nearly tied — strong signal for Data Science and Cloud Computing respectively.
- Students who prefer "Finding and closing security gaps" have the highest average satisfaction, suggesting Cybersecurity students self-select well.
- These task-preference categories directly validate the quiz question weightings used in the interest profiler.

### 12.5 Alternative Electives — Where the Regret Points

In [ ]:
# ── Where would switchers go? ─────────────────────────────────────────────────
switchers = survey[
    survey['alt_elective'].notna() &
    (survey['alt_elective'].str.strip() != '') &
    (~survey['alt_elective'].str.lower().isin(['n/a', 'nan']))
].copy()
switchers['alt_elective'] = switchers['alt_elective'].str.strip()

alt_counts = switchers['alt_elective'].value_counts()

# Flow: from original elective → alt elective
flow = switchers.groupby(['elective', 'alt_elective']).size().reset_index(name='count')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: bar chart of alt elective demand
ax = axes[0]
alt_colors = [SURVEY_COLORS.get(e, '#94a3b8') for e in alt_counts.index]
bars = ax.barh(alt_counts.index[::-1], alt_counts.values[::-1],
               color=alt_colors[::-1], alpha=0.85)
ax.set_xlabel('Number of Students Who Would Switch Here', fontsize=10)
ax.set_title('Most Desired Alternative Electives', fontsize=12, fontweight='bold')
ax.set_xlim(0, alt_counts.max() + 1.5)
for bar in bars:
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width())}', va='center', fontsize=10, fontweight='bold')

# Right: switch flow table
ax2 = axes[1]
ax2.axis('off')
flow_display = flow.sort_values('count', ascending=False)
table_data = [[row['elective'], '→', row['alt_elective'], str(row['count'])]
              for _, row in flow_display.iterrows()]
tbl = ax2.table(
    cellText=table_data,
    colLabels=['From Elective', '', 'To Elective', 'Students'],
    loc='center',
    cellLoc='left'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.4)
ax2.set_title('Switch Flow: From → To', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

**Observations:**
- Cloud Computing is the most-wanted alternative (5 students would switch here), confirming its broad appeal.
- The most common individual switch: Data Science → System Integration, suggesting students who chose Data Science sometimes feel drawn to the engineering / systems side.
- Cloud Computing → Blockchain is an interesting signal: students excited about infrastructure are also drawn to decentralised tech.
- These flow patterns are directly usable to improve quiz recommendations — high-regret pairs should be surfaced as "You may also want to consider..." suggestions.

### 12.6 Quiz Weight Calibration from Survey Data

In [ ]:
# ── Compute satisfaction-based weight multipliers ────────────────────────────
#
# Methodology:
#   1. Compute mean satisfaction per elective from the survey.
#   2. Compute how far each elective's mean is from the overall mean.
#   3. Apply a gentle scaling: multiplier = 1.0 + (avg_sat - overall_mean) * 0.08
#      This gives at most ±15% adjustment. NLP signal still dominates.
#   4. A 'confidence weight' is also computed from had_enough_info — electives
#      where students felt informed before choosing get a small extra boost.
# ──────────────────────────────────────────────────────────────────────────────

SURVEY_TO_MODEL = {
    'Cloud Computing':    'cloud',
    'Cybersecurity':      'cybersecurity',
    'Data Science':       'data_eng',
    'Mobile Development': 'mobile',
    'System Integration': 'system_integration',
    'Quality Assurance':  'qa',
    'Blockchain Development': 'blockchain',
}

overall_sat  = survey['satisfaction'].mean()
overall_info = survey['had_enough_info'].mean()

calibration = (
    survey.groupby('elective')
    .agg(
        avg_sat=('satisfaction', 'mean'),
        avg_info=('had_enough_info', 'mean'),
        n=('satisfaction', 'count'),
        pct_yes=('would_choose_again', lambda x: (x == 'Yes').mean() * 100)
    )
    .reset_index()
)

calibration['sat_multiplier']  = (1.0 + (calibration['avg_sat']  - overall_sat)  * 0.08).round(3)
calibration['info_multiplier'] = (1.0 + (calibration['avg_info'] - overall_info) * 0.04).round(3)
calibration['final_multiplier'] = (calibration['sat_multiplier'] * calibration['info_multiplier']).round(3)
calibration['model_key']       = calibration['elective'].map(SURVEY_TO_MODEL)

print('=== Quiz Weight Calibration Table ===')
print(calibration[['elective','n','avg_sat','pct_yes','sat_multiplier',
                    'info_multiplier','final_multiplier','model_key']]
      .sort_values('final_multiplier', ascending=False)
      .to_string(index=False))

print()
print('Interpretation:')
for _, row in calibration.sort_values('final_multiplier', ascending=False).iterrows():
    direction = '↑ Boost' if row['final_multiplier'] > 1.02 else \
                '↓ Reduce' if row['final_multiplier'] < 0.98 else '= Neutral'
    print(f"  {row['elective']:<25} multiplier={row['final_multiplier']:.3f}  {direction}")

In [ ]:
# ── Visualise the calibration multipliers ────────────────────────────────────
cal_sorted = calibration.sort_values('final_multiplier', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))

bar_colors = [
    '#16a34a' if v > 1.02 else '#dc2626' if v < 0.98 else '#94a3b8'
    for v in cal_sorted['final_multiplier']
]
bars = ax.barh(
    cal_sorted['elective'],
    cal_sorted['final_multiplier'],
    color=bar_colors, alpha=0.88
)
ax.axvline(1.0, color='#0f172a', linestyle='--', linewidth=1.5,
           label='Neutral (×1.000)')
ax.set_xlim(0.85, 1.20)
ax.set_xlabel('Quiz Weight Multiplier', fontsize=11)
ax.set_title('Survey-Calibrated Quiz Weight Multipliers\n'
             '(Green = boost · Red = reduce · Grey = neutral)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
for bar, val in zip(bars, cal_sorted['final_multiplier']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'×{val:.3f}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

**Observations:**
- Electives with above-average satisfaction get a gentle upward multiplier applied to their quiz scores — this makes the recommendation slightly favour specialisations where real students report higher post-choice satisfaction.
- The maximum adjustment is ≈ ±10% so the NLP signal (TF-IDF cosine similarity) remains dominant.
- Data Science scores slightly below average in satisfaction but this may reflect misaligned expectations rather than poor course quality — the calibration is intentionally gentle to avoid over-correcting on small sample sizes.

### 12.7 What Would Have Helped Students Choose?

In [ ]:
# ── Student comments / what would have helped ────────────────────────────────
comments = survey[
    survey['comments'].notna() &
    (~survey['comments'].str.strip().str.lower().isin(['', 'n/a', 'nan']))
][['elective', 'satisfaction', 'would_choose_again', 'comments']].copy()

print(f'Comments with content: {len(comments)}\n')
print('=== Student Comments on What Would Have Helped ===')
for _, row in comments.iterrows():
    stars = '★' * int(row['satisfaction']) + '☆' * (5 - int(row['satisfaction']))
    print(f'  [{row["elective"]}] {stars}  ({row["would_choose_again"]})')
    print(f'  "{row["comments"].strip()[:150]}"')
    print()

# Common themes — keyword frequency in comments
from sklearn.feature_extraction.text import CountVectorizer as CV
comment_vec = CV(stop_words='english', max_features=30, ngram_range=(1,2))
comment_matrix = comment_vec.fit_transform(comments['comments'].fillna(''))
word_freq = dict(zip(comment_vec.get_feature_names_out(),
                     comment_matrix.toarray().sum(axis=0)))
freq_df = pd.DataFrame(list(word_freq.items()), columns=['term','count'])\
            .sort_values('count', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(freq_df['term'][::-1], freq_df['count'][::-1], color='#2563eb', alpha=0.85)
ax.set_xlabel('Frequency in Comments', fontsize=11)
ax.set_title('Most Common Terms in Student Comments\n(What Would Have Helped)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

**Observations:**
- The word *workshop* appears repeatedly — students consistently ask for hands-on demos or sessions from people already in the elective.
- *Information*, *elective*, and *research* are the next most common — students wanted more structured information and wish they had done more research.
- *Careers* and *tools* also appear — students wanted to see what real-world careers and toolchains look like within each elective.
- This is precisely what the Explore Electives page and Job Match feature in Elective Compass provides.

### 12.8 Updated Summary Table with Survey Data

In [ ]:
# ── Merge survey insights with NLP model metrics ─────────────────────────────

# NLP avg similarity scores (already computed in section 10)
avg_sim_scores = sim_df.mean(axis=1).reset_index()
avg_sim_scores.columns = ['syllabus_filename', 'avg_nlp_score']
avg_sim_scores['elective_key'] = avg_sim_scores['syllabus_filename'].str.replace('syllabus_', '')

# Map survey elective names to model keys
survey_summary = (
    calibration[['elective', 'n', 'avg_sat', 'pct_yes', 'final_multiplier', 'model_key']]
    .rename(columns={
        'n':               'survey_n',
        'avg_sat':         'avg_satisfaction',
        'pct_yes':         'pct_would_choose_again',
        'final_multiplier':'quiz_multiplier',
        'model_key':       'elective_key',
    })
)

merged = survey_summary.merge(avg_sim_scores, on='elective_key', how='left')
merged['avg_satisfaction']         = merged['avg_satisfaction'].round(2)
merged['pct_would_choose_again']   = merged['pct_would_choose_again'].round(0).astype(int)
merged['avg_nlp_score']            = merged['avg_nlp_score'].round(4)

print('=== Combined Model + Survey Summary ===')
print(merged[['elective','survey_n','avg_satisfaction','pct_would_choose_again',
              'quiz_multiplier','avg_nlp_score']]
      .sort_values('avg_satisfaction', ascending=False)
      .to_string(index=False))

## 13. Final Summary — All Findings Combined

| Dimension | Finding |
|---|---|
| **Corpus** | 7 syllabi + 70 job descriptions + 36 student survey responses |
| **Model accuracy** | TF-IDF correctly matches majority of jobs to their own elective |
| **Most NLP-distinctive** | Blockchain — highest vocabulary uniqueness |
| **Most NLP-overlapping** | Cloud ↔ Data Engineering — shared tooling |
| **Highest satisfaction** | Mobile Development (5.0, n=2); Cybersecurity (4.33, n=6) |
| **Highest regret** | Data Science — most "No, I'd switch" responses |
| **Lowest pre-choice confidence** | Mobile Development (2.0 avg) — students discovered fit post-choice |
| **Most-wanted alternative** | Cloud Computing — 5 students would switch here |
| **Core student need** | Workshops, hands-on demos, career path visibility — what Elective Compass provides |
| **Quiz calibration** | Gentle ±8% multipliers applied based on real satisfaction scores |

---

### What the survey data adds to the model

The NLP model tells you which elective *matches your language* — the words you use align with which syllabus.
The survey data tells you which elective *students are actually happy with* after living it.
The calibration layer blends both signals: **text similarity × satisfaction multiplier = final recommendation score.**

---

*Notebook complete. All data sources integrated: syllabi, job descriptions, and real student survey responses.*